In [1]:
# Import Statments:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Adding Device Management:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is available and set as device.")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available and set as device.")
else:
    device = torch.device("cpu")
    print("Using CPU device.")


MPS is available and set as device.


In [2]:
# Curating General Peptide Data from PeptideAtlas:
with open('Generalized_Peptide_Data.txt', 'r') as f:
    general_peptides = f.read().splitlines()

print(f'{len(general_peptides)} general peptides loaded | Sample peptide: {general_peptides[5]}')


285668 general peptides loaded | Sample peptide: CLQSGTLFR


In [3]:
max_len = 27

general_peptide_data = []

# Converting Peptides to Fixed-Length with Special Tokens:
for peptide in general_peptides:

    peptide = ['<CLS>'] + list(peptide) + ['<BED>']

    for _ in range(max_len - len(peptide)):
        peptide.append('<PAD>')

    general_peptide_data.append(peptide)

print(f'Sample peptide: {''.join(general_peptide_data[5])}')
print(f'Peptide Length: {len(general_peptide_data[5])} tokens')


Sample peptide: <CLS>CLQSGTLFR<BED><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
Peptide Length: 27 tokens


In [4]:
# Creating Vocabulary for Amino Acids + Special Tokens:
amino_acids = ['<CLS>',
 'A',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'K',
 'L',
 'M',
 'N',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'V',
 'W',
 'Y',
 '<BED>',
 '<PAD>']

vocab_size = len(amino_acids)

aatoi = {aa:i for i,aa in enumerate(amino_acids)}
itoaa = {i:aa for i,aa in enumerate(amino_acids)}

encode = lambda p: [aatoi[aa] for aa in p]
decode = lambda l: [itoaa[i] for i in l]

print(f'Sample peptide: {''.join(general_peptide_data[5])}')
print(f'Tokenized peptide: {encode(general_peptide_data[5])}')


Sample peptide: <CLS>CLQSGTLFR<BED><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
Tokenized peptide: [0, 2, 10, 14, 16, 6, 17, 10, 5, 15, 21, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22]


In [5]:
# Creating General Peptide Tensor:
tokenized_peptides = []

for peptide in general_peptide_data:

    tokenized_peptides.append(encode(peptide))

general_peptide_tensor = torch.tensor(tokenized_peptides, dtype=torch.long).to(device)

print(f'General peptide tensor shape: {list(general_peptide_tensor.shape)}')


General peptide tensor shape: [285668, 27]


In [6]:
# Creating Training/Validation Split:
train_val_split = int(0.9 * len(general_peptide_tensor))

general_peptide_train = general_peptide_tensor[:train_val_split]
general_peptide_val = general_peptide_tensor[train_val_split:]

print(f'Number of training peptides: {len(general_peptide_train)}')
print(f'Number of validation peptides: {len(general_peptide_val)}')


Number of training peptides: 257101
Number of validation peptides: 28567


In [7]:
# Creating Batches of General Peptide Data:
batch_size = 1

padding_tensor = torch.tensor([22], dtype=torch.long).to(device)

def get_batch(split):

    data = general_peptide_train if split == 'train' else general_peptide_val

    ix = torch.randint(len(data), (batch_size,))

    x = torch.stack([data[i] for i in ix])
    y = torch.stack([torch.cat([data[i][1:], padding_tensor]) for i in ix])

    return x, y

x, y = get_batch('train')

print(f'Sample index: {x.tolist()}')
print(f'Sample target: {y.tolist()}')
    

Sample index: [[0, 4, 4, 4, 14, 1, 17, 4, 17, 14, 13, 8, 18, 20, 6, 14, 13, 14, 10, 11, 10, 21, 22, 22, 22, 22, 22]]
Sample target: [[4, 4, 4, 14, 1, 17, 4, 17, 14, 13, 8, 18, 20, 6, 14, 13, 14, 10, 11, 10, 21, 22, 22, 22, 22, 22, 22]]


In [8]:
# Model Parameters:
n_embd = 256
head_size = 32
n_layer = 8
n_head = 8
batch_size = 32
block_size = 27
dropout = 0.1

# Attention Head:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()

        # K,Q,V Matrices:
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Buffer Matrix and Dropout Layer:
        self.register_buffer('tril', torch.tril(torch.ones([block_size, block_size])))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5

        # Padding Masking:
        is_padding = (x == 0).all(dim=-1)  # Shape: [B, T]
        padding_mask = is_padding.unsqueeze(1).expand(-1, T, -1)  # Shape: [B, T, T]
        wei = wei.masked_fill(padding_mask, float('-inf'))

        # Causal Masking:
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Adjust Embeddings:
        v = self.value(x)
        out = wei @ v
        return out

# Multi-Headed Attention:
class MultiHeadedAttention(nn.Module):

    def __init__(self, head_size, n_head):
        super().__init__()

        # List of Attention Heads:
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])

        # Projection and Dropout Layers:
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# Multi-Layer Perceptron:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        # Linear Layers, ReLU and Dropout:
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * 4),
            nn.ReLU(),
            nn.Linear(n_embd * 4, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Self-Attention/FeedForward Block:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head

        # Self-Attention/FeedForward:
        self.sa = MultiHeadedAttention(head_size, n_head)
        self.ffwd = FeedForward(n_embd)

        # Layer Normalization:
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    # Residual Blocks and Self-Attention/FeedForward:
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# AMP Transformer Model:
class AMPTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # Token and Positional Embedding Tables:
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Block Layers:
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

        # Layer Normalization and Unembedding:
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Regression Head:
        self.mic_head = nn.Linear(n_embd, 1)

    def forward(self, idx, targets=None):
        B,T = idx.shape

        # Embeddings:
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        # Creating Logits after Forward Pass:
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        # Determining Loss via Cross Entropy:
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.reshape(B*T, C)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=22)

        return logits, loss

    def get_representation(self, idx):
        B,T = idx.shape
            
        # Embeddings:
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        x = x[:, 0, :] 
            
        return x

    def predict_mic(self, idx):

        # Predict MIC Value with Regression Head:
        representation = self.get_representation(idx)
        mic_value = self.mic_head(representation).squeeze()
        
        return mic_value

    def generate(self, idx):

        # Generate New Peptide Sequence:
        while True:
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_new = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_new], dim=1)
            if idx_new.item() == 21:
                break

        return idx
    
# Initializing Model:
model = AMPTransformer().to(device)

# Creating Optimizer:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

print(f'Model initialized with {sum(p.numel() for p in model.parameters())} parameters')


Model initialized with 6331416 parameters


In [9]:
from tqdm import tqdm

general_train_losses = []
general_val_losses = []

# Creating Training Loop:
pbar = tqdm(range(5000), desc="training")

for step in pbar:

    x, y = get_batch('train')
    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    general_train_losses.append(loss)

    pbar.set_postfix({'loss': loss.item()})

    with torch.no_grad():
        x, y = get_batch('val')
        _, loss = model(x, y)
        
    general_val_losses.append(loss)

print(f'Loss value after training: {loss.item()}')


training: 100%|██████████████████| 5000/5000 [07:39<00:00, 10.87it/s, loss=2.73]

Loss value after training: 2.7400100231170654


In [10]:
# Curating AMP Data from dbAMP:
with open('AMP_Data.txt', 'r') as f:
    amps = f.read().splitlines()

# Curating non-AMP Data from UniProt:
with open('non-AMP_Data.txt', 'r') as f:
    non_amps = f.read().splitlines()

print(f'{len(amps)} AMPs loaded | Sample AMP: {amps[5]}')
print(f'{len(non_amps)} non-AMPs loaded | Sample non-AMP: {non_amps[5]}')


18158 AMPs loaded | Sample AMP: AAGMGFFGAR
17452 non-AMPs loaded | Sample non-AMP: SYSFHDDSKRERSAKSRYTV


In [11]:
amp_data = []
non_amp_data = []

# Converting AMPs to Fixed-Length with Special Tokens:
for amp in amps:

    amp = ['<CLS>'] + list(amp) + ['<BED>']

    for _ in range(max_len - len(amp)):
        amp.append('<PAD>')

    amp_data.append(amp)

# Converting non-AMPs to Fixed-Length with Special Tokens:
for non_amp in non_amps:

    non_amp = ['<CLS>'] + list(non_amp) + ['<BED>']

    for _ in range(max_len - len(non_amp)):
        non_amp.append('<PAD>')

    non_amp_data.append(non_amp)

print(f'Sample AMP: {''.join(amp_data[5])}')
print(f'Sample non_AMP: {''.join(non_amp_data[5])}')
print(f'AMP Length: {len(amp_data[5])} tokens')
print(f'non_AMP Length: {len(non_amp_data[5])} tokens')


Sample AMP: <CLS>AAGMGFFGAR<BED><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD>
Sample non_AMP: <CLS>SYSFHDDSKRERSAKSRYTV<BED><PAD><PAD><PAD><PAD><PAD>
AMP Length: 27 tokens
non_AMP Length: 27 tokens


In [12]:
# Creating AMP Tensor:
tokenized_amps = []

for amp in amp_data:

    tokenized_amps.append(encode(amp))

amp_tensor = torch.tensor(tokenized_amps, dtype=torch.long).to(device)

# Creating non-AMP Tensor:
tokenized_non_amps = []

for non_amp in non_amp_data:

    tokenized_non_amps.append(encode(non_amp))

non_amp_tensor = torch.tensor(tokenized_non_amps, dtype=torch.long).to(device)

amp_tensor = amp_tensor[:16384]
non_amp_tensor = non_amp_tensor[:16384]

print(f'AMP tensor shape: {list(amp_tensor.shape)}')
print(f'non-AMP tensor shape: {list(non_amp_tensor.shape)}')


AMP tensor shape: [16384, 27]
non-AMP tensor shape: [16384, 27]


In [13]:
# Creating Training/Validation Split:
train_val_split = int(0.9 * len(amp_tensor))

amp_train = amp_tensor[:train_val_split]
amp_val = amp_tensor[train_val_split:]

non_amp_train = non_amp_tensor[:train_val_split]
non_amp_val = non_amp_tensor[train_val_split:]

print(f'Number of training AMPs: {len(amp_train)}')
print(f'Number of training non-AMPs: {len(non_amp_train)}')
print(f'Number of validation AMPs: {len(amp_val)}')
print(f'Number of validation non-AMPs: {len(non_amp_val)}')


Number of training AMPs: 14745
Number of training non-AMPs: 14745
Number of validation AMPs: 1639
Number of validation non-AMPs: 1639


In [14]:
# Creating Batches of AMP and non-AMP Data:
batch_size = 32

def get_contrastive_batch(split):
    
    amp_data = amp_train if split == 'train' else amp_val
    non_amp_data = non_amp_train if split == 'train' else non_amp_val
    
    ix = torch.randint(len(amp_data), (batch_size//2,))
    
    amp_sequences = torch.stack([amp_data[i] for i in ix])
    non_amp_sequences = torch.stack([non_amp_data[i] for i in ix])
    
    # Combining AMP and non-AMP Sequences:
    all_sequences = torch.cat([amp_sequences, non_amp_sequences], dim=0)
    
    # Creating Targets for Autoregressive Training:
    targets = torch.cat([
        all_sequences[:, 1:], 
        torch.full((batch_size, 1), 22).to(device)  
    ], dim=1)
    
    return all_sequences, targets, amp_sequences, non_amp_sequences

all_sequences, targets, amp_sequences, non_amp_sequences = get_contrastive_batch('train')

print(f'all_sequences shape: {list(all_sequences.shape)}')
print(f'targets shape: {list(targets.shape)}')
print(f'amp_sequences shape: {list(amp_sequences.shape)}')
print(f'non_amp_sequences shape: {list(non_amp_sequences.shape)}')


all_sequences shape: [32, 27]
targets shape: [32, 27]
amp_sequences shape: [16, 27]
non_amp_sequences shape: [16, 27]


In [15]:
# Creating Contrastive Loss Function:
contrastive_train_losses = []
contrastive_val_losses = []

lambda_weight = 0.1

def contrastive_loss(model, amp_batch, non_amp_batch, margin=1.0):
    
    # Creating AMP and non-AMP Representations:
    amp_repr = model.get_representation(amp_batch)
    non_amp_repr = model.get_representation(non_amp_batch)
    
    # Normalizing Probability Distributions:
    amp_repr = F.softmax(amp_repr, dim=-1)
    non_amp_repr = F.softmax(non_amp_repr, dim=-1)
    
    # Creating KL divergence:
    kl_div = F.kl_div(non_amp_repr.log(), amp_repr, reduction='batchmean')
    
    # Calculating Loss:
    loss = torch.max(torch.tensor(0.0).to(device), margin - kl_div) ** 2
    
    return loss

# Creating Contrastive Training Loop:
pbar = tqdm(range(5000), desc="training")

for step in pbar:

    # Contrastive Loss + Auto-Regressive Loss:
    sequences, targets, amps, non_amps = get_contrastive_batch('train')
    logits, arlm_loss = model(sequences, targets)
    contr_loss = contrastive_loss(model, amps, non_amps)
    total_loss = arlm_loss + lambda_weight * contr_loss

    optimizer.zero_grad(set_to_none=True)
    total_loss.backward()
    optimizer.step()

    contrastive_train_losses.append(total_loss)

    pbar.set_postfix({'loss': total_loss.item()})

    with torch.no_grad():
        sequences, targets, amps, non_amps = get_contrastive_batch('val')
        logits, arlm_loss = model(sequences, targets)
        contr_loss = contrastive_loss(model, amps, non_amps)
        total_loss = arlm_loss + lambda_weight * contr_loss

    contrastive_val_losses.append(total_loss)
    
print(f'Loss value after training: {total_loss.item()}')


training: 100%|███████████████████| 5000/5000 [54:55<00:00,  1.52it/s, loss=1.7]

Loss value after training: 2.203169822692871


In [16]:
# Curating MIC Value Data:
import pandas as pd

df = pd.read_csv('MIC_Value_Data.csv')

df.tail()


,sequence,value
5087,GCWSTVLGGLKKFAKGGLEAIVNPK,0.903090
5088,KEEQIGKSSTRGRKSSRRKK,1.000000
5089,AAKKLSKLLKTLLKLL,0.778151
5090,KKWKKFIKKIGIGAVLTTPGAKK,0.301030
5091,LNKGAILKHIIK,2.000000


In [17]:
mic_sequence_data = []
mic_value_data = df['value'].values

# Converting MIC Data Sequences to Fixed-Length with Special Tokens:
for sequence in df['sequence']:

    sequence = ['<CLS>'] + list(sequence) + ['<BED>']

    for _ in range(max_len - len(sequence)):
        sequence.append('<PAD>')

    mic_sequence_data.append(sequence)

print(f'Sample sequence: {''.join(mic_sequence_data[5])}')
print(f'sequence Length: {len(mic_sequence_data[5])} tokens')
print(f'MIC value: {mic_value_data[5]} ug/ml')


Sample sequence: <CLS>GWGSIFKHGRHAAKHIGHAAVNHYL<BED>
sequence Length: 27 tokens
MIC value: 1.806179974 ug/ml


In [18]:
# Creating MIC Sequence Tensor:
tokenized_sequences = []

for sequence in mic_sequence_data:

    tokenized_sequences.append(encode(sequence))

mic_sequence_tensor = torch.tensor(tokenized_sequences, dtype=torch.long).to(device)
mic_value_tensor = torch.tensor(mic_value_data, dtype=torch.float32).to(device)

# Normalizing MIC Value Tensor:
mic_mean = mic_value_tensor.mean()
mic_std = mic_value_tensor.std()
mic_value_tensor = (mic_value_tensor - mic_mean) / mic_std

print(f'MIC sequence tensor shape: {list(mic_sequence_tensor.shape)}')
print(f'MIC value tensor shape: {list(mic_value_tensor.shape)}')


MIC sequence tensor shape: [5092, 27]
MIC value tensor shape: [5092]


In [19]:
# Creating Training/Validation Split:
train_val_split = int(0.9 * len(mic_sequence_data))

mic_sequence_train = mic_sequence_tensor[:train_val_split]
mic_sequence_val = mic_sequence_tensor[train_val_split:]

mic_value_train = mic_value_tensor[:train_val_split]
mic_value_val = mic_value_tensor[train_val_split:]

print(f'Number of training sequences: {len(mic_sequence_train)}')
print(f'Number of training values: {len(mic_value_train)}')
print(f'Number of validation sequences: {len(mic_sequence_val)}')
print(f'Number of validation values: {len(mic_value_val)}')


Number of training sequences: 4582
Number of training values: 4582
Number of validation sequences: 510
Number of validation values: 510


In [20]:
# Creating Batches of MIC Data:
batch_size = 1

def get_mic_batch(split):

    sequence_data = mic_sequence_train if split == 'train' else mic_sequence_val
    value_data = mic_value_train if split == 'train' else mic_value_val
    
    ix = torch.randint(len(sequence_data), (batch_size,))
    
    x = torch.stack([sequence_data[i] for i in ix])
    y = torch.stack([value_data[i] for i in ix])
    
    return x, y

x, y = get_mic_batch('train')

print(f'Sample index: {x.tolist()}')
print(f'Sample target: {y.tolist()}')


Sample index: [[0, 9, 1, 1, 2, 1, 1, 9, 2, 10, 19, 15, 21, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22, 22]]
Sample target: [1.2137387990951538]


In [21]:
# Creating MIC Prediction Training Loop:
mic_train_losses = []
mic_val_losses = []

batch_size = 32

pbar = tqdm(range(5000), desc="training")

for step in pbar:

    # Calculating MIC loss:
    x, y = get_mic_batch('train')
    MIC_predictions = model.predict_mic(x)
    mic_loss = F.mse_loss(MIC_predictions, y)
    
    optimizer.zero_grad(set_to_none=True)
    mic_loss.backward()
    optimizer.step()

    mic_train_losses.append(mic_loss)

    pbar.set_postfix({'loss': mic_loss.item()})

    with torch.no_grad():
        x, y = get_mic_batch('val')
        MIC_predictions = model.predict_mic(x)
        mic_loss = F.mse_loss(MIC_predictions, y)

    mic_val_losses.append(mic_loss)

print(f'Loss value after training: {mic_loss.item()}')
    

training: 100%|█████████████████| 5000/5000 [08:26<00:00,  9.88it/s, loss=0.972]

Loss value after training: 1.7784557342529297


In [30]:
# Creating Novel Data:
n_peptides = 10
index = 0

while index < n_peptides:

    new_peptide = model.generate(torch.zeros([1, 1], dtype=torch.long).to(device))[0].tolist()

    display_sequence = new_peptide

    for _ in range(max_len - len(new_peptide)):
        new_peptide = new_peptide + [22]

    new_peptide = torch.tensor([new_peptide], dtype=torch.long).to(device)

    MIC_prediction = 10 ** (model.predict_mic(new_peptide) * mic_std + mic_mean)

    if (len(display_sequence) > max_len or len(display_sequence) <= 6 or MIC_prediction >= 20):
        continue

    index += 1

    print(f'{index}.\nSequence: {''.join(decode(display_sequence[1:-1]))}')

    print(f'MIC value: {MIC_prediction} ug/ml\n')
    

1.
Sequence: DTKSNGGTTGPQAGG
MIC value: 18.617332458496094 ug/ml

2.
Sequence: GVILVEPVDD
MIC value: 19.262319564819336 ug/ml

3.
Sequence: RKCVQLERNSCRWSV
MIC value: 18.398521423339844 ug/ml

4.
Sequence: AKIMSHEEFPVLTELREVA
MIC value: 18.463003158569336 ug/ml

5.
Sequence: PVKKGKRSSRRKKRFFGMAESF
MIC value: 18.158592224121094 ug/ml

6.
Sequence: FGSCDKVRHCVFTGSFVNMRW
MIC value: 17.993602752685547 ug/ml

7.
Sequence: ADAVCRFVGMCWSVPESVSRRAT
MIC value: 18.567079544067383 ug/ml

8.
Sequence: TYVNGLEIKVHQLQYFIMA
MIC value: 18.113815307617188 ug/ml

9.
Sequence: IWDFVW
MIC value: 18.587013244628906 ug/ml

10.
Sequence: QKNHREKAVNITQEGCE
MIC value: 19.178945541381836 ug/ml

